# Local collection status and verification

Read-only, no API calls or sign-in. Run All in **primusEnv**. For a stable final validation, interrupt/wait for download notebooks to finish first. F1 and weather archives are reported separately; a weather backlog does not mean F1 telemetry is absent.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'data/archive/store.py').is_file() and (p / 'primusEnv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the F1 Primus AI project.')
expected = (ROOT / 'primusEnv/Scripts/python.exe').resolve()
if Path(sys.executable).resolve() != expected:
    raise RuntimeError(f'Select the primusEnv kernel: {expected}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Python:', sys.executable)


In [ ]:
DRY_RUN = False
VERIFY_FILE_HASHES = True  # Reads local files; may take time for the entire archive.


In [ ]:
import sqlite3
from data.archive.store import file_hash
from data.archive.reader import ArchiveReader
from data.archive.collect import worker_lock
import pyarrow.parquet as pq

if not DRY_RUN:
    for directory in (ROOT / '.local/f1_2025_2026', ROOT / '.local/f1_weather'):
        database = directory / 'f1.sqlite3'
        print('Archive:', directory)
        if not database.is_file():
            print('Not collected yet.')
            continue
        try:
            lock = worker_lock(directory)
            lock.__enter__()
        except RuntimeError:
            print('Collector active: wait for it to finish or interrupt it, then rerun validation.')
            continue
        try:
            connection = sqlite3.connect(database.resolve().as_uri() + '?mode=ro', uri=True)
            connection.row_factory = sqlite3.Row
            try:
                assert connection.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
                assert not connection.execute('PRAGMA foreign_key_check').fetchall()
                for table in ('events', 'sessions', 'session_versions', 'weather_series', 'weather_observations'):
                    print(table, connection.execute(f'SELECT count(*) FROM {table}').fetchone()[0])
                if VERIFY_FILE_HASHES:
                    rows = connection.execute('SELECT relative_path,file_sha256,row_count FROM dataset_files').fetchall()
                    for row in rows:
                        path = (directory / row['relative_path']).resolve()
                        assert path.is_relative_to(directory.resolve()) and not path.is_symlink()
                        assert file_hash(path) == row['file_sha256'], f'Hash mismatch: {path}'
                        assert pq.ParquetFile(path).metadata.num_rows == row['row_count']
                    sources = connection.execute('SELECT raw_path,payload_sha256 FROM source_objects').fetchall()
                    for row in sources:
                        path = (directory / row['raw_path']).resolve()
                        assert path.is_relative_to(directory.resolve()) and not path.is_symlink()
                        assert file_hash(path) == row['payload_sha256'], f'Source hash mismatch: {path}'
                    print(f'Verified {len(rows)} manifest Parquet files and {len(sources)} source objects.')
                    weather_tasks = connection.execute("SELECT checkpoint_json FROM collection_tasks WHERE dataset_kind='weather' AND status IN ('complete','partial')").fetchall()
                    for task in weather_tasks:
                        state = json.loads(task['checkpoint_json'])
                        if 'parquet_path' not in state:
                            continue
                        path = (directory / state['parquet_path']).resolve()
                        assert path.is_relative_to(directory.resolve()) and not path.is_symlink()
                        assert file_hash(path) == state['parquet_sha256'], f'Weather export hash mismatch: {path}'
                        assert pq.ParquetFile(path).metadata.num_rows == state['observations']
                        count = connection.execute('SELECT count(*) FROM weather_observations WHERE series_id=? AND source_id=?', (state['series_id'], state['source_id'])).fetchone()[0]
                        assert count == state['observations'], f'Weather SQL coverage mismatch: {path}'
                    print(f'Verified {len(weather_tasks)} weather task exports where present.')
                tasks = pd.read_sql_query('SELECT requested_scope,dataset_kind,status,next_retry_utc,last_error_code FROM collection_tasks', connection)
                if not tasks.empty:
                    display(tasks)
            finally:
                connection.close()
            sessions = ArchiveReader(directory).available_sessions()
            if sessions:
                display(pd.DataFrame(sessions)[['season','event_name','full_name','session_id']])
            for progress in sorted(directory.glob('progress-*.json')):
                print('Progress:', progress.name)
                print(json.dumps(json.loads(progress.read_text(encoding='utf8')), indent=2, default=str))
        finally:
            lock.__exit__(None, None, None)
else:
    print('Offline preview only; validation reads skipped.')


## Weather exports
Weather task checkpoints also record each event’s Parquet export path/hash and session links. The weather runner validates/rebuilds those exports on resume. This report checks the shared SQL/source/manifest archive; published session counts are not expected for independent weather-only series.
